# Prepare our flight and weather data

Run each cell from top to bottom. We will look at the original files, keep the data we need, and save one combined CSV.

We are using **2024 departures from Knoxville, Nashville, Memphis, and Chattanooga**.


## 1. Choose the months
Use `list(range(1, 13))` for the full year, or `[1]` for a quicker January run.

In [ ]:
from pathlib import Path
import sys
import importlib
import pandas as pd
from IPython.display import display

# Find our project folder and load its supporting code.
ROOT = next((folder for folder in [Path.cwd(), *Path.cwd().parents]
             if (folder / "Prepare_Data.ipynb").exists()
             and (folder / "data_helpers.py").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the FlightDelaysAndWeather folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import data_helpers
importlib.reload(data_helpers)
from data_helpers import ensure_data, load_flights, load_weather, match_weather

YEAR = 2024
MONTHS = list(range(1, 13))  # Change to [1] to try January first.
assert MONTHS and all(isinstance(m, int) and 1 <= m <= 12 for m in MONTHS)
assert len(MONTHS) == len(set(MONTHS))
print("Year:", YEAR, "| Months:", MONTHS)


## 2. Get the files
Missing files download automatically. Files we already have are reused. The first full-year download may take a few minutes.

In [ ]:
flight_paths, weather_paths, sources = ensure_data(ROOT, YEAR, MONTHS)
print(f"Ready: {len(flight_paths)} flight files and {len(weather_paths)} weather files.")


## 3. Look at the raw data
Each flight row is one scheduled flight. Each weather row is a reading from an airport weather station.
The weather values contain codes; we will turn those into readable numbers in step 5.

In [ ]:
print("Flight sample:")
raw_flights = pd.read_csv(flight_paths[0], nrows=5)
display(raw_flights[["FlightDate", "Origin", "Dest", "CRSDepTime", "DepDelay", "Cancelled"]])

print("Knoxville weather sample:")
raw_weather = pd.read_csv(weather_paths["TYS"], nrows=5, dtype="string")
display(raw_weather[["DATE", "TMP", "WND", "VIS"]])


## 4. Keep our flights
Keep departures from TYS, BNA, MEM, and CHA for the selected months. Keep cancellations too.
Times are put on the same clock so we can match them with weather later.

In [ ]:
flights, flight_counts, duplicates_removed = load_flights(flight_paths, YEAR, MONTHS)
print(f"Read {flight_counts.raw_rows.sum():,} nationwide flight records.")
print(f"Kept {len(flights):,} flights from our four airports.")
print(f"Removed {duplicates_removed} exact duplicate records.")
display(flights.groupby("Origin").agg(
    flights=("flight_id", "size"), cancellations=("Cancelled", "sum")))
display(flights[["FlightDate", "Origin", "Dest", "DepDelay", "Cancelled"]].head())


## 5. Clean the weather
Keep airport weather reports and turn the codes into temperature, wind speed, visibility, and cloud ceiling.
Missing or questionable readings stay blank. Blank does not mean zero.
Rain, snow, and thunderstorm codes are kept for later work; they are not classified yet.

In [ ]:
weather, weather_counts = load_weather(weather_paths, YEAR, MONTHS)
print(f"Read {weather_counts.raw_rows.sum():,} weather reports.")
print(f"Kept {len(weather):,} reports for matching.")
display(weather[["Origin", "weather_time_utc", "TMP_raw", "temperature_c",
                 "wind_speed_mps", "visibility_m"]].head())
print("Missing readings:")
display(weather[["temperature_c", "wind_speed_mps", "visibility_m"]]
        .isna().sum().to_frame("missing"))


## 6. Match each flight with weather
Use the most recent report from the **same airport**, at or before scheduled departure, within 90 minutes.
Flights without a match stay in the table with blank weather. Each row should still represent one flight.

In [ ]:
combined = match_weather(flights, weather, tolerance_minutes=90)
summary = combined.groupby("Origin").agg(
    flights=("flight_id", "size"), weather_matches=("weather_matched", "sum"))
summary["no_weather_match"] = summary["flights"] - summary["weather_matches"]
display(summary)
display(combined[["FlightDate", "Origin", "scheduled_departure_utc",
                  "weather_time_utc", "weather_age_minutes", "DepDelay", "Cancelled"]].head())


## 7. Save the combined CSV
Check that no flights or cancellations were lost, then save the result under `data/processed/`.
January and full-year runs have different filenames. Original downloads stay unchanged.

In [ ]:
assert len(combined) == len(flights)
assert combined["flight_id"].is_unique
assert combined["Cancelled"].sum() == flights["Cancelled"].sum()
matched = combined["weather_matched"]
assert combined.loc[matched, "weather_age_minutes"].between(0, 90).all()
print("Checks passed: flight counts, cancellations, and weather timing.")
print("Possible duplicate flights to review:", int(combined["possible_duplicate_flight"].sum()))
print("Matched reports with no usable core readings:",
      int((matched & combined["valid_weather_measurements"].eq(0)).sum()))

output_dir = ROOT / "data/processed"
output_dir.mkdir(parents=True, exist_ok=True)
period = "full_year" if sorted(MONTHS) == list(range(1, 13)) else "months_" + "_".join(f"{m:02}" for m in sorted(MONTHS))
output_path = output_dir / f"flights_weather_{YEAR}_{period}.csv"
combined.to_csv(output_path, index=False)
assert len(pd.read_csv(output_path, usecols=["flight_id"])) == len(combined)
print(f"Saved {len(combined):,} flights to {output_path.relative_to(ROOT)}")


## What comes next?
Define weather groups, compare delays and cancellations, and make charts. These matches show weather near a flight's departure; they do not prove why it was delayed.
